<a href="https://colab.research.google.com/github/Ans365332/6may-file-example/blob/main/Webhook_and_Queue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Listen to Queue and Webhooks

In [2]:
!pip install -qU langchain langchain-google-genai google-generativeai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.7/561.7 kB 36.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.


In [3]:
# LLM Setup

In [5]:
import getpass
import os

if "GOOGLE_API_KEY" not in os.environ:
  os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your  API Key:")


from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model= "gemini-2.5-flash",temperature=0)



In [8]:
import json
import time
import uuid
import random
import queue
import hashlib
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable, List , Dict , Any , Optional

In [9]:
# Target-1 ::  Listen to Webhook and Queues

In [12]:
class SimulatedEventQueue:

  def __init__(self):
    self._queue: "queue.Queue" = queue.Queue()

  def push_event(self, event_type:str, payload:Dict[str,Any], event_id:Optional[str]=None):
    event = {
        "event_id": event_id or str(uuid.uuid4()),
        "event_type": event_type,
        "payload": payload,
        "recieved_at": datetime.now().strftime("%Y-%M-%D  %H:%M:%S")

    }
    self.queue.put(event)
    print(f"[listener] event received: type='{event_type}' id={event['event_id'][:8]}...")
    return event

  def listen_and_drain(self):
    events = []
    while not self._queue.empty():
      events.append(self._queue.get())
    return events

  def is_empty(self):
    return self._queue.empty()

  event_queue = SimulatedEventQueue()
  print("Queue Set Up is ready.....")


Queue Set Up is ready.....


In [ ]:
#### Execute Workflow on trigger

In [13]:
class WorkflowRegistry:

  def __init__(self):
    self._workflows: Dict[str , Callable] = {}

  def register(self, event_type:str , workflow_fn: Callable):
    self._workflows[event_type] = workflow_fn
    print(f"[Workflow-registery] registered workflow for event_type = '{event_type}' ")

  def get(self, event_type: str):
    return self._workflows.get(event_type)

  def known_event_types(self):
    return list(self._workflows.keys())